In [ ]:
from bs4 import BeautifulSoup
import requests
from datetime import date, timedelta, datetime

In [ ]:
aylar={'yanvar': 1,
       'fevral': 2,
       'mart': 3,
       'aprel': 4,
       'may': 5,
       'iyun': 6,
       'iyul': 7,
       'avqust': 8,
       'sentyabr': 9,
       'oktyabr': 10,
       'noyabr': 11,
       'dekabr': 12}

sozler = [
    "sığorta",
    "patent",
    "əmtəə nişanı",
    "qiymətli metallar",
    "qiymətli daşlar",
    "qiymətli kağızlar",
    "inzibati xəta",
    "Məcəllə",
    "pensiya",
    "büdcə sistemi",
    "mühasibat uçotu",
    "dövlət satınalmaları",
    "süni intellekt",
    "rəqəmsallaşma",
    "rəqəmsal inkişaf",
    "biznes",
    "bank",
    "investisiya",
    "rəqabət",
    "informasiya",
    "əmək haqqı",
    "enerji resursları",
    "xalis mənfəət",
    "nəzarət zərfi",
    "sahibkarlıq",
    "sosial müdafiə",
    "güzəştli maliyyələşdirmə",
    "vergi",
    "Mərkəzi Bank",
    "qaz təchizatı",
    "kibercinayət",
    "cinayət",
    "notariat",
    "informasiya sistemi",
    "reklam",
    "daşınmaz əmlak",
    "ipoteka",
    "istehlakçıların hüquqları",
    "texniki təhlükəsizlik",
    "lisenziya",
    "tikinti",
    "bədbəxt hadisə",
    "peşə xəstəlikləri",
    "istehsalat",
    "antiinhisar",
    "maliyyə monitorinqi",
    "kredit",
    "nağdsız",
    "elektron hökumət",
    "bank hesabı",
    "icra haqqında",
    "terrorçuluğun maliyyələşdirilməsi",
    "kredit zəmanət fondu",
    "ödəniş sistemləri",
    "ödəniş xidmətləri",
    "emitent",
    "kredit təşkilatı",
    "valyuta tənzimi",
    "restrukturizasiya",
    "maliyyə hesabatı",
    "maliyyə bazarı",
]

notsozler = [
    "minatəmizləmə",
    "dövlət qulluqçusu",
    "dövlət qulluğu",
    "kinematoqrafiya",
    "kosmonavt",
    "kosmik fəza",
]

sozler2 = [
    "ödəniş sistemləri",
    "emitent",
    "ekvayer",
    "investisiya",
    "faiz",
    "maliyyə",
    "uçot dərəcəsi",
    "ipoteka",
    "kredit",
    "ƏDV",
    "vergi ödəyiciləri",
    "hüquqi şəxs",
    "kredit riskləri",
    "maliyyə institutları",
    "nağd ödənişlər",
    "nağdsız ödənişlər",
    "tarif",
    "mobil ödənişlər",
    "POS terminal",
    "internet bankçılıq",
    "sığorta",
    "lizinq",
    "audit",
    "mühasibatlıq",
    "büdcə",
    "aktivlər",
    "kapital",
    "likvidlik",
    "bank hesabı",
    "eyniləşdirmə",
    "gücləndirilmiş elektron imza",
    "GMA",
]

notsozler2 = [
    "səyyar qəbulu",
    "ev təsərrüfatları",
    "İşçi qrup",
    "İclas",
    "təlim",
    "müsabiqə",
    "Təcrübə Proqramı",
    "mini-futbol",
    "intellektual",
]

def parse_az_date(date_text):
    parts = date_text.replace(',', '').split()

    day = int(parts[0])
    month = aylar[parts[1].lower()]
    year = int(parts[2])
    hour, minute = map(int, parts[3].split(':'))

    return datetime(year, month, day, hour, minute)

In [ ]:
import time
import random
import pandas as pd

def parse_az_date(text):
    text  = text.strip()
    parts = text.split()
    day   = int(parts[0])
    month = aylar[parts[1].lower()]
    year  = int(parts[2][:4])
    return date(year, month, day)

today   = date.today()
cutoff  = today - timedelta(days=90)
results = []
i = 1

while True:
    url = f"https://president.az/az/documents/category/decrees/{i}"
    soup = BeautifulSoup(requests.get(url).text, "html.parser")
    time.sleep(random.uniform(1, 3))

    items = soup.find_all("article", class_="news-text")
    if not items:
        break

    stop = False

    for item in items:
        title_tag = item.find("a", class_="news-text_body")
        date_tag = item.find("span", class_="category_date")

        if not title_tag or not date_tag:
            continue

        title = title_tag.get_text(strip=True)
        date_text = date_tag.get_text(strip=True).lower()

        try:
            item_date = parse_az_date(date_text)
        except:
            continue

        if item_date < cutoff:
            stop = True
            break

        detail_url = title_tag["href"]
        if not detail_url.startswith("http"):
            detail_url = f"https://president.az{detail_url}"

        detail_soup = BeautifulSoup(requests.get(detail_url).text, "html.parser")
        time.sleep(random.uniform(1, 3))

        body_div = detail_soup.find("div", class_="news-text-block")
        body = body_div.get_text(separator=" ", strip=True) if body_div else ""

        basliq = [kw for kw in sozler2 if kw.lower() in title.lower() and all(nkw.lower() not in title.lower() for nkw in notsozler2)]
        xeber = [kw for kw in sozler2 if kw.lower() in body.lower()  and all(nkw.lower() not in body.lower()  for nkw in notsozler2)]

        if basliq or xeber:
            print(item_date.strftime("%d.%m.%Y"))
            print(f"  Basliq : {title}")
            if basliq:
                print(f"  Soz1   : {basliq}")
            if xeber:
                print(f"  Soz2   : {xeber}")
                print(f"  Mezmun : {body[:300]}")
            print()
            results.append({"Basliq": title, "Xeber": body, 'Tarix': item_date, "Basliqdaki Sozler": basliq, 'Xeberdeki Sozler': xeber})

    if stop:
        break

    i += 1

df = pd.DataFrame(results)
df.to_excel("decrees.xlsx", index=False)

08.06.2026
  Basliq : Azərbaycan Respublikası Prezidentinin “Azərbaycan Respublikası Kənd Təsərrüfatı Nazirliyi yanında Aqrar Kredit və İnkişaf Agentliyi haqqında Əsasnamənin təsdiq edilməsi haqqında” 2015-ci il 23 iyul tarixli 571 nömrəli, “Kənd təsərrüfatına dövlət dəstəyinin və aqrar sahədə lizinq fəaliyyətinin təkmilləşdirilməsi haqqında” 2018-ci il 19 dekabr tarixli 413 nömrəli, “Aqrar sahədə yeni subsidiya mexanizminin yaradılması haqqında” 2019-cu il 27 iyun tarixli 759 nömrəli və “Elektron kənd təsərrüfatı” informasiya sistemi haqqında Əsasnamə”nin təsdiq edilməsi barədə” 2019-cu il 23 dekabr tarixli 897 nömrəli fərmanlarında dəyişiklik edilməsi barədə     Azərbaycan Respublikası Prezidentinin  FərmanıAzərbaycan Respublikası Konstitusiyasının 109-cu maddəsinin 32-ci bəndini rəhbər tutaraq qərara alıram:1. Azərbaycan Respublikası Prezidentinin 2015-ci il 23 iyul tarixli 571 nömrəli Fərmanı (Azərbaycan Respublikasının Qanunvericilik Toplusu,...08 iyun 2026, 17:20
  Soz1   : ['kre

In [ ]:
df

,Basliq,Xeber,Tarix,Basliqdaki Sozler,Xeberdeki Sozler
0,Azərbaycan Respublikası Prezidentinin “Azərbay...,Azərbaycan Respublikası Konstitusiyasının 109-...,2026-06-08,"[kredit, lizinq]","[faiz, maliyyə, kredit, hüquqi şəxs, sığorta, ..."


In [ ]:
from datetime import datetime, date, timedelta
import requests
from bs4 import BeautifulSoup
import time
import random
import pandas as pd

today = date.today()
cutoff  = today - timedelta(days=90)
results = []
i = 1

while True:
    url = f"https://mia.gov.az/az/news/archive/{i}"
    soup = BeautifulSoup(requests.get(url).text, "html.parser")
    time.sleep(random.uniform(1, 3))

    items = soup.find_all("div", class_="press-news-item")
    if not items:
        break

    stop = False

    for item in items:
        title_tag = item.find("a")
        date_tag  = item.find("span")

        if not title_tag or not date_tag:
            continue

        title = title_tag.get_text(strip=True)

        try:
            item_date = datetime.strptime(
                date_tag.get_text(strip=True), "%d.%m.%Y"
            ).date()
        except ValueError:
            continue

        if item_date < cutoff:
            stop = True
            break

        detail_url = title_tag["href"]
        if not detail_url.startswith("http"):
            detail_url = f"https://mia.gov.az{detail_url}"

        detail_soup = BeautifulSoup(requests.get(detail_url).text, "html.parser")
        time.sleep(random.uniform(1, 3))

        body_div = detail_soup.find("div", class_="news-inside-content")
        body = body_div.get_text(separator=" ", strip=True) if body_div else ""

        basliq = [kw for kw in sozler2 if kw.lower() in title.lower() and all(nkw.lower() not in title.lower() for nkw in notsozler2)]
        xeber = [kw for kw in sozler2 if kw.lower() in body.lower()  and all(nkw.lower() not in body.lower()  for nkw in notsozler2)]

        if basliq or xeber:
            print(item_date.strftime("%d.%m.%Y"))
            print(f"  Basliq : {title}")
            if basliq:
                print(f"  Soz1   : {basliq}")
            if xeber:
                print(f"  Soz2   : {xeber}")
                print(f"  Mezmun : {body[:300]}")
            print()
            results.append({"Basliq": title, "Xeber": body, 'Tarix': item_date, "Basliqdaki Sozler": basliq, 'Xeberdeki Sozler': xeber})

    if stop:
        break

    i += 1

df = pd.DataFrame(results)
df.to_excel("mia_news.xlsx", index=False)

04.06.2026
  Basliq : Nazir Vilayət Eyvazov Qazaxda vətəndaş qəbulu, Ağstafada sıra baxışı keçirib
  Soz2   : ['ƏDV']
  Mezmun : Nazir Vilayət Eyvazov Qazaxda vətəndaş qəbulu, Ağstafada sıra baxışı keçirib 04.06.2026 Azərbaycan Respublikasının Prezidenti cənab İlham Əliyevin tapşırığına və bölgələr üzrə vətəndaş qəbulu cədvəlinə uyğun olaraq daxili işlər naziri general-polkovnik Vilayət Eyvazov iyunun 3-də və 4-də Qazax rayon

07.05.2026
  Basliq : Nazir Vilayət Eyvazov İmişlidə vətəndaş qəbulu və sıra baxışı keçirib
  Soz2   : ['ƏDV']
  Mezmun : Nazir Vilayət Eyvazov İmişlidə vətəndaş qəbulu və sıra baxışı keçirib 07.05.2026 Azərbaycan Respublikasının Prezidenti cənab İlham Əliyevin tapşırığına və bölgələr üzrə vətəndaş qəbulu cədvəlinə uyğun olaraq daxili işlər naziri general-polkovnik Vilayət Eyvazov mayın 6-da və 7-də İmişli rayonunda nö

01.04.2026
  Basliq : Nazir Vilayət Eyvazov Cəlilabadda növbəti vətəndaş qəbulu və sıra baxışı keçirib
  Soz2   : ['ƏDV']
  Mezmun : Nazir Vilayə

In [ ]:
import time
import random
import pandas as pd

today = date.today()
cutoff = today - timedelta(days=90)
results = []
i = 1

while True:
    url = f"https://www.bfb.az/press-relizler?page={i}"
    soup = BeautifulSoup(requests.get(url).text, "html.parser")
    time.sleep(random.uniform(1, 3))

    items = soup.find_all("div", class_="col-md-6")
    if not items:
        break

    stop = False

    for item in items:
        title_tag = item.find("a", class_="post_title")
        date_tag  = item.find("div", class_="date")

        if not title_tag or not date_tag:
            continue

        title = title_tag.get_text(strip=True)
        date_text = date_tag.get_text(strip=True).lower()

        try:
            day, month_name, year = date_text.split()
            item_date = date(int(year), aylar[month_name], int(day))
        except:
            continue

        if item_date < cutoff:
            stop = True
            break

        detail_url = title_tag["href"]
        if not detail_url.startswith("http"):
            detail_url = f"https://www.bfb.az{detail_url}"

        detail_soup = BeautifulSoup(requests.get(detail_url).text, "html.parser")
        time.sleep(random.uniform(1, 3))

        body_div = detail_soup.find("div", class_="text_editor")
        body = body_div.get_text(separator=" ", strip=True) if body_div else ""

        basliq = [kw for kw in sozler2 if kw.lower() in title.lower() and all(nkw.lower() not in title.lower() for nkw in notsozler2)]
        xeber  = [kw for kw in sozler2 if kw.lower() in body.lower()  and all(nkw.lower() not in body.lower()  for nkw in notsozler2)]

        if basliq or xeber:
            print(item_date.strftime("%d.%m.%Y"))
            print(f"  Basliq : {title}")
            if basliq:
                print(f"  Soz1  : {basliq}")
            if xeber:
                print(f"  Soz2  : {xeber}")
                print(f"  Mezmun: {body[:300]}")
            print()
            results.append({"Basliq": title, "Xeber": body, 'Tarix': item_date, "Basliqdaki Sozler": basliq, 'Xeberdeki Sozler': xeber})

    if stop:
        break

    i += 1

df = pd.DataFrame(results)
df.to_excel("news.xlsx", index=False)


08.06.2026
  Basliq : "AZƏRMAŞ GROUP" MMC-nin istiqrazlarının yerləşdirilməsi üzrə hərrac yekunlaşmışdır
  Soz2  : ['emitent', 'faiz']
  Mezmun: 2 iyun 2026-cı il tarixində Bakı Fond Birjasının (BFB) Listinq Komitəsinin qərarı ilə "AZƏRMAŞ GROUP" MMC-nin istiqrazlarının ticarətə buraxılması üçün “Ticarətə İcazə” qərarı verilmişdir. Qeyd olunan AZ2002027301 ISIN kodlu istiqrazların yerləşdirilməsi üzrə hərrac 08.06.2026-cı il tarixində yekunl

04.06.2026
  Basliq : Azərbaycan Respublikasının Mərkəzi Bankının 252 günlük Notlarının yerləşdirilməsi üzrə hərrac keçirilmişdir
  Soz2  : ['emitent']
  Mezmun: 4 iyun 2026-cı il tarixində Bakı Fond Birjasında Azərbaycan Respublikasının Mərkəzi Bankının (ARMB) AZ2648024704 dövlət qeydiyyat nömrəli 5,000,000 manat həcmində 252 gün tədavül müddətli notlarının yerləşdirilməsi üzrə hərrac keçirilmişdir. Notlar üzrə məlumat aşağıdakı kimidir: Emitent Azərbaycan 

04.06.2026
  Basliq : Azərbaycan Respublikasının Mərkəzi Bankının 84 günlük Notlarının ye

In [ ]:
df

,Basliq,Xeber
0,Azərbaycan Respublikasının Mərkəzi Bankının 25...,4 iyun 2026-cı il tarixində Bakı Fond Birjasın...
1,Azərbaycan Respublikasının Mərkəzi Bankının 84...,4 iyun 2026-cı il tarixində Bakı Fond Birjasın...
2,"""BQ"" ASC-nin səhmlərinin abunə yazılışı üsulu ...",3 iyun 2026-cı il tarixində Bakı Fond Birjasın...
3,Azərbaycan Respublikasının Mərkəzi Bankının 16...,3 iyun 2026-cı il tarixində Bakı Fond Birjasın...
4,Azərbaycan Respublikasının Mərkəzi Bankının 28...,3 iyun 2026-cı il tarixində Bakı Fond Birjasın...
5,Azərbaycan Respublikasının Mərkəzi Bankının No...,3 və 4 iyun 2026-cı il tarixlərində Bakı Fond ...
6,"""AZƏRMAŞ GROUP"" MMC-nin istiqrazları üçün “Tic...",2 iyun 2026-cı il tarixində Bakı Fond Birjasın...
7,Azərbaycan Respublikasının İpoteka və Kredit Z...,Azərbaycan Respublikasının İpoteka və Kredit Z...
8,Azərbaycan Respublikasının İpoteka və Kredit Z...,Azərbaycan Respublikasının İpoteka və Kredit Z...
9,Azərbaycan Respublikasının İpoteka və Kredit Z...,Azərbaycan Respublikasının İpoteka və Kredit Z...


In [ ]:
from datetime import date, timedelta
import requests
from bs4 import BeautifulSoup
import time
import random
import pandas as pd

def parse_az_date(text):
    parts = text.strip().split()
    day   = int(parts[0])
    month = aylar[parts[1].lower()]
    year  = int(parts[2].split("-")[0])
    return date(year, month, day)

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/124.0 Safari/537.36",
    "Accept-Language": "az,en;q=0.9",
    "Referer": "https://fhn.gov.az/",
}

today      = date.today()
start_date = today - timedelta(days=90)
results = []

for back in range(0, 4):
    if back == 0:
        target_date = today
    else:
        target_date = today.replace(day=1) - timedelta(days=30 * back)

    year  = target_date.year
    month = target_date.month

    url = f"https://fhn.gov.az/az/gundelik-xronika?month={month}&year={year}"
    soup = BeautifulSoup(requests.get(url, headers=headers).text, "html.parser")
    time.sleep(random.uniform(1, 3))

    items = soup.find_all("div", class_="col-md-6")

    for item in items:
        a_tag = item.find("a", class_="daily__item")
        if not a_tag:
            continue

        h3_tag = a_tag.find("h3")
        p_tag = a_tag.find("p")

        if not h3_tag or not p_tag:
            continue

        date_text = h3_tag.get_text(strip=True)
        body = p_tag.get_text(separator=" ", strip=True)

        try:
            item_date = parse_az_date(date_text)
        except Exception as e:
            print(f"Date parse error: '{date_text}' → {e}")
            continue

        if not (start_date <= item_date <= today):
            continue

        basliq = [kw for kw in sozler2 if kw.lower() in date_text.lower() and all(nkw.lower() not in date_text.lower() for nkw in notsozler2)]
        xeber = [kw for kw in sozler2 if kw.lower() in body.lower()       and all(nkw.lower() not in body.lower()       for nkw in notsozler2)]

        if basliq or xeber:
            print(item_date.strftime("%d.%m.%Y"))
            print(f"  Basliq : {date_text}")
            if basliq:
                print(f"  Soz1   : {basliq}")
            if xeber:
                print(f"  Soz2   : {xeber}")
                print(f"  Mezmun : {body[:300]}")
            print()
            results.append({"Basliq": title, "Xeber": body, 'Tarix': item_date, "Basliqdaki Sozler": basliq, 'Xeberdeki Sozler': xeber})

results_sorted = sorted(results, key=lambda x: x["Basliq"], reverse=True)
df = pd.DataFrame(results_sorted)
df.to_excel("fhn_news.xlsx", index=False)

In [ ]:
df

""
